# LD二次展開＋運動RWA と、補正付き第一サイドバンドの比較

同じ駆動・初期熱状態・散逸条件で、次の2モデルの二量子ビットチャネルを比較します。

| モデル | LD展開と運動RWA | 数値計算に残る運動演算子 |
|:--|:--|:--|
| **A: LD二次展開＋RWA** | 指数を絶対次数 $O(\eta^2)$ まで展開し、第一サイドバンドの近共鳴項を残す | $B_A=a$ |
| **B: 補正付き第一サイドバンド** | 指数の $O(\eta^3)$ までに含まれる第一サイドバンド成分を残す | $B_B=a-\frac{\eta^2}{2}(N+1)a=a(1-\frac{\eta^2}{2}N)$ |

**モデルAは、運動RWA後には通常の線形MSハミルトニアンになります。**
二次のキャリア補正と第二サイドバンド項は、第一サイドバンド駆動では非共鳴のため除かれます。
これは「LD二次までの全項を残し、非共鳴項も時間発展させるモデルA」の計算ではありません。
両モデルとも光学RWA・運動RWAを適用済みです。通常のRWAで除いた非共鳴項の高次有効補正は加えていません。

リポジトリの符号・規格化をそのまま使います。
$$H_M(t)=\hbar g S_x\left[B_M e^{-i\delta t}+B_M^\dagger e^{i\delta t}\right],
  \quad S_x=X\otimes I+I\otimes X,\quad M\in\{A,B\}.$$
ここで $g$ はコードのパラメータ `A`（モデル名Aとは別）で、先頭のLD因子を含む有効結合です。
$$T=2\pi/\delta,\qquad U_\star=\exp(+i\pi XX/4).$$
離調の符号も既存コードに合わせています。
矩形パルス・同じ振幅を比較し、モデル別の再較正は行いません。

上から順に実行します。既定では5温度点 × 6散逸条件 × 2モデル = **60チャネル**を、各16入力で再構成します。
まず短く試す場合は設定セルの `NBAR_VALUES` と `CASE_NAMES` を減らしてください。
同一設定の完了済み点は、このノートブック専用のキャッシュから再開できます。

In [1]:
import os
import tempfile
from pathlib import Path

# 数値ライブラリのimport前に、ワーカー内のスレッド数を固定する。
for name in (
    "OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS",
):
    os.environ[name] = "1"
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "ms_ab_matplotlib"))

import hashlib
import importlib
import json
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import qutip as qp
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "ms_gate_functions.py").is_file():
    raise RuntimeError("このノートブックはMS_gate_Hiraiプロジェクト直下から実行してください。")
import ms_gate_functions as mg
import pauli_channel_distance as pcd
mg = importlib.reload(mg)
pcd = importlib.reload(pcd)


## 共通パラメータと散逸条件

`single_mode_ms_gate_infidelity.ipynb` と同じ基準値です。
$g=0.125$, $\delta=0.5$, $\eta=0.1$, 物理ゲート時間 $100\,\mu\mathrm{s}$、時間点数500。
$\bar n=0.01,1,2,3,4$。有効結合と離調は従来のシミュレーション単位です。
散逸率は物理単位から既存関数が換算します。

Heating: $\Gamma_h=10\,\mathrm{s}^{-1}$、motional dephasing: $\Gamma_m=18\,\mathrm{s}^{-1}$、
spin dephasing: $T_2^*=0.3\,\mathrm{s}$、Rayleigh/Raman: $3/1\,\mathrm{s}^{-1}$。
Heatingの $\mathcal D[a]+\mathcal D[a^\dagger]$、各スピン散逸の規格化も既存コードに従います。

`no_noise`、4種類の散逸を1種類ずつ有効にする条件、`all_noise` を比較します。
各温度の運動基底次元は両モデル・全散逸条件で共通です。
`PHONON_DIM_SCALE` を上げれば別設定として収束を比較できます。初期熱分布の尾確率とCP/TP診断だけでは、数値収束の証明にはなりません。

In [2]:
NBAR_VALUES = [0.01, 1.0, 2.0, 3.0, 4.0]
CASE_NAMES = [
    "no_noise", "motional_heating", "motional_dephasing",
    "spin_dephasing", "photon_scattering", "all_noise",
]
WORKERS = min(6, os.cpu_count() or 1)
PHONON_DIM_SCALE = 1.0
REUSE_CACHE = True
OUTPUT_ROOT = ROOT / "results/ld_second_order_rwa_vs_corrected_sideband"
DISPLAY_CASE = "motional_dephasing"

BASE_PARAMETERS = {
    "A": 0.125,
    "delta": 0.5,
    "rho0": 0.0,
    "time_points": 500,
    "t_gate_phys": 100e-6,
    "heating_rate_phys": 10.0,
    "dephasing_rate_phys": 18.0,
    "T2_star": 0.3,
    "rayleigh_rate_phys": 3.0,
    "raman_rate_phys": 1.0,
    "eta": 0.1,
    "laser_intensity_fluctuation": 0.0,
    "laser_detuning_fluctuation": 0.0,
    "laser_rotation_angle_fluctuation": 0.0,
    "laser_noise_samples": 1,
    "laser_noise_seed": 1234,
    "solver_max_step": None,
}
MODELS = {
    "A": {"label": "A: LD through eta^2 + motional RWA", "use_full_order": False},
    "B": {"label": "B: corrected first sideband", "use_full_order": True},
}
NOISE_OFF = {
    "heating_rate_phys": 0.0, "dephasing_rate_phys": 0.0,
    "T2_star": np.inf, "rayleigh_rate_phys": 0.0, "raman_rate_phys": 0.0,
}
NOISE_KEYS = {
    "no_noise": (),
    "motional_heating": ("heating_rate_phys",),
    "motional_dephasing": ("dephasing_rate_phys",),
    "spin_dephasing": ("T2_star",),
    "photon_scattering": ("rayleigh_rate_phys", "raman_rate_phys"),
    "all_noise": tuple(NOISE_OFF),
}


In [3]:
assert NBAR_VALUES and len(set(NBAR_VALUES)) == len(NBAR_VALUES)
assert np.all(np.isfinite(NBAR_VALUES)) and np.all(np.asarray(NBAR_VALUES) >= 0)
assert CASE_NAMES and len(set(CASE_NAMES)) == len(CASE_NAMES)
assert set(CASE_NAMES) <= set(NOISE_KEYS)
assert WORKERS >= 1 and np.isfinite(PHONON_DIM_SCALE) and PHONON_DIM_SCALE >= 1
assert BASE_PARAMETERS["delta"] > 0  # 既存の +XX ターゲット規約

U_IDEAL = mg.ideal_ms_gate(phi=np.pi / 4)
SUPER_DIMS = qp.to_super(U_IDEAL).dims
PAULI_LABELS, PAULIS = pcd.pauli_basis()
RHO_00 = qp.ket2dm(qp.tensor(qp.basis(2, 0), qp.basis(2, 0)))
RHO_BELL = (U_IDEAL * RHO_00 * U_IDEAL.dag()).full()
CHI_IDENTITY = np.zeros((16, 16), dtype=complex)
CHI_IDENTITY[0, 0] = 1
alpha_max = 2 * abs(BASE_PARAMETERS["A"] / BASE_PARAMETERS["delta"])
DIMENSIONS = {
    float(n): int(np.ceil(PHONON_DIM_SCALE * mg.estimate_phonon_dim(float(n), alpha_max)))
    for n in NBAR_VALUES
}
case_parameters = {}
for case in CASE_NAMES:
    parameters = {**BASE_PARAMETERS, **NOISE_OFF}
    for name in NOISE_KEYS[case]:
        parameters[name] = BASE_PARAMETERS[name]
    case_parameters[case] = parameters

display(pd.DataFrame([
    {"nbar": n, "phonon_dim": DIMENSIONS[n],
     "initial_thermal_tail": (n / (1 + n)) ** DIMENSIONS[n]}
    for n in DIMENSIONS
]))
display(pd.DataFrame([
    {"case": case, "heating_s^-1": p["heating_rate_phys"],
     "motional_dephasing_s^-1": p["dephasing_rate_phys"],
     "spin_dephasing_s^-1": 1 / p["T2_star"],
     "rayleigh_s^-1": p["rayleigh_rate_phys"], "raman_s^-1": p["raman_rate_phys"]}
    for case, p in case_parameters.items()
]))
print(f"Planned: {2 * len(CASE_NAMES) * len(NBAR_VALUES)} channels, 16 inputs each")


,nbar,phonon_dim,initial_thermal_tail
0,0.01,20,8.195445e-41
1,1.00,25,2.980232e-08
2,2.00,36,4.578410e-07
3,3.00,51,4.247412e-07
4,4.00,67,3.213876e-07


,case,heating_s^-1,motional_dephasing_s^-1,spin_dephasing_s^-1,rayleigh_s^-1,raman_s^-1
0,no_noise,0.0,0.0,0.000000,0.0,0.0
1,motional_heating,10.0,0.0,0.000000,0.0,0.0
2,motional_dephasing,0.0,18.0,0.000000,0.0,0.0
3,spin_dephasing,0.0,0.0,3.333333,0.0,0.0
4,photon_scattering,0.0,0.0,0.000000,3.0,1.0
5,all_noise,10.0,18.0,3.333333,3.0,1.0


Planned: 60 channels, 16 inputs each


## チャネル再構成と評価量

16入力から得た実ゲートを $\mathcal E$ とし、これまでと同じ **ゲート後の誤差**
$\mathcal N=\mathcal E\circ\mathcal U_\star^{-1}$ を解析します（`undo_before_actual`）。
平均ゲート忠実度は $F_{\rm avg}=(4F_{\rm pro}+1)/5$ です。

$\chi$ は通常のPauli基底、$\operatorname{Tr}\chi=1$ の規約です。
QuTiPのChoi表現から直接変換し、奇数個のYを含むPauli演算子の符号規約に依存しないようにします。
Pauli twirling はこの**誤差チャネル**の $\chi$ を対角化します。

- `infidelity`: $1-F_{\rm avg}$（負の数値残差もクリップしません）。
- `chi_offdiag_fro`: 非対角成分のFrobeniusノルム。確率ではありません。
- `twirl_choi_trace_distance`: 正規化Choi行列のtrace distance。diamond distanceそのものではありません。
- `bell_survival` / `bell_survival_twirl`: 同じ理想Bell状態に対する $\mathcal N$ / $\mathcal N_{\rm PT}$ の生存確率。
  これは特定状態の診断で、一般のQEC回路に対する近似誤差ではありません。

保存キーに全パラメータ・打ち切り・モデル・QuTiPバージョン・計算モジュールのハッシュを含めます。
以前の `ld2` フォルダからは読み込みません。数値チャネルへのCPTP射影や再規格化はしません。

In [4]:
def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (tuple, list)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, float) and not np.isfinite(value):
        return str(value)
    return value

def encoded(value):
    return json.dumps(json_safe(value), sort_keys=True, ensure_ascii=False, allow_nan=False)

ENGINE_HASH = hashlib.sha256(Path(mg.__file__).read_bytes()).hexdigest()
RUN_CONFIG = {
    "schema": "ld-ab-rwa-v1", "models": MODELS,
    "base_parameters": BASE_PARAMETERS, "case_parameters": case_parameters,
    "nbar_values": NBAR_VALUES, "phonon_dimensions": DIMENSIONS,
    "phonon_dim_scale": PHONON_DIM_SCALE, "workers": WORKERS,
    "error_channel_convention": "undo_before_actual",
    "target": "exp(+i*pi*XX/4)", "amplitude_calibrated": False,
    "qutip_version": qp.__version__, "numpy_version": np.__version__,
    "ms_gate_functions_sha256": ENGINE_HASH,
    "pauli_channel_distance_sha256": hashlib.sha256(Path(pcd.__file__).read_bytes()).hexdigest(),
}
RUN_ID = hashlib.sha256(encoded(RUN_CONFIG).encode()).hexdigest()[:16]
OUTPUT_DIR = OUTPUT_ROOT / RUN_ID
CACHE_DIR = OUTPUT_ROOT / "point_cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "config.json").write_text(encoded(RUN_CONFIG), encoding="utf-8")

def point_channel(model, case, nbar):
    parameters = {
        **case_parameters[case], "n_bar_list": [float(nbar)],
        "phonon_dim_override": DIMENSIONS[float(nbar)],
        "use_full_order": MODELS[model]["use_full_order"],
        "parallel_workers": WORKERS, "show_progress": False,
    }
    payload = {
        "schema": "ld-ab-rwa-channel-v1", "model": model,
        "parameters": parameters, "qutip_version": qp.__version__,
        "numpy_version": np.__version__, "engine_sha256": ENGINE_HASH,
    }
    signature = encoded(payload)
    key = hashlib.sha256(signature.encode()).hexdigest()
    path = CACHE_DIR / f"{model}_{case}_{key[:24]}.npz"
    if REUSE_CACHE and path.exists():
        with np.load(path, allow_pickle=False) as saved:
            if str(saved["signature"].item()) != signature:
                raise RuntimeError(f"Cache signature mismatch: {path}")
            matrix = saved["superoperator"].copy()
        if matrix.shape != (16, 16) or not np.isfinite(matrix).all():
            raise RuntimeError(f"Invalid cached channel: {path}")
        return qp.Qobj(matrix, dims=SUPER_DIMS, superrep="super"), True

    result = mg.generate_process_channels(**parameters)
    channel = result["S_qobj_list"][0]
    if not np.isfinite(channel.full()).all():
        raise RuntimeError("Non-finite channel; no cache was written.")
    temporary = path.with_suffix(".tmp")
    with temporary.open("wb") as stream:
        np.savez_compressed(stream, signature=signature, superoperator=channel.full())
    temporary.replace(path)
    return channel, False

def analyze_channel(channel):
    error = mg.remove_ideal_gate_from_channel(
        channel, ideal_unitary=U_IDEAL, convention="undo_before_actual"
    )
    choi = qp.to_choi(error).full()
    chi = pcd.choi_to_chi(choi, PAULIS)
    if not np.allclose(pcd.chi_to_choi(chi, PAULIS), choi, atol=1e-11, rtol=1e-11):
        raise RuntimeError("Choi/Pauli representation mismatch")
    ptm = pcd.chi_to_ptm(chi, PAULIS)
    twirled = pcd.pauli_twirl_chi(chi)
    bell_actual = float(np.trace(RHO_BELL @ pcd.apply_chi(chi, RHO_BELL, PAULIS)).real)
    bell_twirl = float(np.trace(RHO_BELL @ pcd.apply_chi(twirled, RHO_BELL, PAULIS)).real)
    direct_output = qp.vector_to_operator(channel * qp.operator_to_vector(RHO_00)).full()
    bell_direct = float(np.trace(RHO_BELL @ direct_output).real)
    if abs(bell_actual - bell_direct) > 1e-10:
        raise RuntimeError("Post-gate error-channel composition mismatch")
    F_pro = float(qp.process_fidelity(channel, U_IDEAL))
    F_avg = (4 * F_pro + 1) / 5
    if abs((1 - F_avg) - 4 / 5 * (1 - chi[0, 0].real)) > 1e-10:
        raise RuntimeError("Fidelity/chi convention mismatch")
    diagnostics = mg.choi_physicality_metrics(channel)
    values = {
        "F_process": F_pro, "F_avg": F_avg, "infidelity": 1 - F_avg,
        "chi_offdiag_fro": float(np.linalg.norm(chi - twirled)),
        "twirl_choi_trace_distance": pcd.trace_norm(pcd.chi_to_choi(chi - twirled, PAULIS)) / 8,
        "bell_survival": bell_actual, "bell_survival_twirl": bell_twirl,
        "bell_twirl_gap": bell_actual - bell_twirl,
        "min_choi_eigenvalue": diagnostics["min_choi_eigenvalue"],
        "tp_frobenius_error": diagnostics["tp_frobenius_error"],
    }
    return values, {"channel": channel, "chi": chi, "ptm": ptm, "choi": choi}


In [5]:
rows = []
channels = {}
started = time.perf_counter()
for case in CASE_NAMES:
    for nbar in NBAR_VALUES:
        for model in MODELS:
            point_started = time.perf_counter()
            print(f"{model} / {case}: nbar={nbar:g}, Nv={DIMENSIONS[float(nbar)]}", flush=True)
            channel, from_cache = point_channel(model, case, nbar)
            values, data = analyze_channel(channel)
            channels[(model, case, float(nbar))] = data
            rows.append({
                "model": model, "case": case, "nbar": float(nbar),
                "phonon_dim": DIMENSIONS[float(nbar)],
                "use_full_order": MODELS[model]["use_full_order"],
                "from_cache": from_cache, "seconds_this_run": time.perf_counter() - point_started,
                **values,
            })
            # 中断時にも、完了済み点を確認できる。
            pd.DataFrame(rows).to_csv(OUTPUT_DIR / "partial_summary.csv", index=False)

summary = pd.DataFrame(rows).sort_values(["case", "nbar", "model"]).reset_index(drop=True)
if not np.isfinite(summary.select_dtypes(include=np.number).to_numpy()).all():
    raise RuntimeError("Non-finite diagnostics; inspect partial results.")
if (summary["min_choi_eigenvalue"] < -1e-8).any() or (summary["tp_frobenius_error"] > 1e-8).any():
    warnings.warn("CP/TP診断に残差があります。打ち切り・数値設定を確認してください。")
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
display(summary[["model", "case", "nbar", "infidelity", "chi_offdiag_fro", "bell_twirl_gap"]])
print(f"Finished: {len(summary)} channels in {time.perf_counter() - started:.1f} s")


A / no_noise: nbar=0.01, Nv=20
B / no_noise: nbar=0.01, Nv=20
A / no_noise: nbar=1, Nv=25
B / no_noise: nbar=1, Nv=25
A / no_noise: nbar=2, Nv=36
B / no_noise: nbar=2, Nv=36
A / no_noise: nbar=3, Nv=51
B / no_noise: nbar=3, Nv=51
A / no_noise: nbar=4, Nv=67
B / no_noise: nbar=4, Nv=67
A / motional_heating: nbar=0.01, Nv=20
B / motional_heating: nbar=0.01, Nv=20
A / motional_heating: nbar=1, Nv=25
B / motional_heating: nbar=1, Nv=25
A / motional_heating: nbar=2, Nv=36
B / motional_heating: nbar=2, Nv=36
A / motional_heating: nbar=3, Nv=51
B / motional_heating: nbar=3, Nv=51
A / motional_heating: nbar=4, Nv=67
B / motional_heating: nbar=4, Nv=67
A / motional_dephasing: nbar=0.01, Nv=20
B / motional_dephasing: nbar=0.01, Nv=20
A / motional_dephasing: nbar=1, Nv=25
B / motional_dephasing: nbar=1, Nv=25
A / motional_dephasing: nbar=2, Nv=36
B / motional_dephasing: nbar=2, Nv=36
A / motional_dephasing: nbar=3, Nv=51
B / motional_dephasing: nbar=3, Nv=51
A / motional_dephasing: nbar=4, Nv=67


/tmp/ipykernel_3699/3609961231.py:26: UserWarning: CP/TP診断に残差があります。打ち切り・数値設定を確認してください。
  warnings.warn("CP/TP診断に残差があります。打ち切り・数値設定を確認してください。")


,model,case,nbar,infidelity,chi_offdiag_fro,bell_twirl_gap
0,A,all_noise,0.01,1.725952e-03,0.000966,4.260787e-10
1,B,all_noise,0.01,1.946533e-03,0.019406,-1.478478e-09
2,A,all_noise,1.00,2.435361e-03,0.001844,-6.998326e-10
3,B,all_noise,1.00,3.644387e-03,0.040592,-2.288127e-09
4,A,all_noise,2.00,3.151344e-03,0.002729,4.846137e-10
5,B,all_noise,2.00,5.980756e-03,0.061419,7.279760e-09
6,A,all_noise,3.00,3.854127e-03,0.003600,-1.610112e-09
7,B,all_noise,3.00,8.877659e-03,0.081608,1.416083e-09
8,A,all_noise,4.00,4.556160e-03,0.004467,-3.206542e-10
9,B,all_noise,4.00,1.228080e-02,0.101106,8.559330e-09


Finished: 60 channels in 3046.3 s


## AとBの差

$\Delta I=I_B-I_A$ と $\|R_B-R_A\|_F$ を比較します。
`AB_choi_trace_distance` は $\|J_B-J_A\|_1/(2d)$（$d=4$）で、diamond distanceの下界です。
忠実度の差が小さくてもチャネルの差が小さいとは限りません。
ここでの差は**同じ散逸条件でのモデル差**であり、「散逸だけが生んだ誤差」ではありません。

In [ ]:
comparison_rows = []
indexed = summary.set_index(["model", "case", "nbar"])
for case in CASE_NAMES:
    for nbar in NBAR_VALUES:
        a = channels[("A", case, float(nbar))]
        b = channels[("B", case, float(nbar))]
        row_a = indexed.loc[("A", case, float(nbar))]
        row_b = indexed.loc[("B", case, float(nbar))]
        comparison_rows.append({
            "case": case, "nbar": float(nbar), "phonon_dim": DIMENSIONS[float(nbar)],
            "infidelity_A": row_a["infidelity"], "infidelity_B": row_b["infidelity"],
            "infidelity_B_minus_A": row_b["infidelity"] - row_a["infidelity"],
            "AB_ptm_fro": float(np.linalg.norm(b["ptm"] - a["ptm"])),
            "AB_choi_trace_distance": pcd.trace_norm(b["choi"] - a["choi"]) / 8,
            "bell_survival_B_minus_A": row_b["bell_survival"] - row_a["bell_survival"],
        })
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
np.savez_compressed(
    OUTPUT_DIR / "channels.npz",
    model=summary["model"].to_numpy(dtype=str), case=summary["case"].to_numpy(dtype=str),
    nbar=summary["nbar"].to_numpy(), pauli_labels=np.asarray(PAULI_LABELS),
    superoperators=np.stack([channels[(r.model, r.case, r.nbar)]["channel"].full() for r in summary.itertuples()]),
    chi_error=np.stack([channels[(r.model, r.case, r.nbar)]["chi"] for r in summary.itertuples()]),
    ptm_error=np.stack([channels[(r.model, r.case, r.nbar)]["ptm"] for r in summary.itertuples()]),
)
display(comparison)

def case_axes():
    ncols = 2
    nrows = (len(CASE_NAMES) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.4 * nrows), squeeze=False)
    for ax in axes.flat[len(CASE_NAMES):]:
        ax.set_visible(False)
    return fig, axes

MODEL_STYLE = {"A": ("o-", "#0072B2"), "B": ("s--", "#D55E00")}
fig, axes = case_axes()
for ax, case in zip(axes.flat, CASE_NAMES):
    for model, (style, color) in MODEL_STYLE.items():
        data = summary.loc[(summary["case"] == case) & (summary["model"] == model)].sort_values("nbar")
        ax.plot(data["nbar"], data["infidelity"], style, color=color, label=MODELS[model]["label"])
    ax.set(title=case.replace("_", " "), xlabel=r"Initial $\bar n$", ylabel=r"$1-F_{\rm avg}$")
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax.grid(alpha=.25)
    ax.legend(fontsize=8)
fig.suptitle("Model comparison at identical drive amplitude and noise rates")
fig.tight_layout(rect=(0, 0, 1, .97))
fig.savefig(OUTPUT_DIR / "infidelity_comparison.png", dpi=180)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for case in CASE_NAMES:
    data = comparison.loc[comparison["case"] == case].sort_values("nbar")
    axes[0].plot(data["nbar"], data["infidelity_B_minus_A"], "o-", label=case.replace("_", " "))
    axes[1].plot(data["nbar"], data["AB_choi_trace_distance"], "o-", label=case.replace("_", " "))
axes[0].set_ylabel(r"$(1-F_{\rm avg})_B-(1-F_{\rm avg})_A$")
axes[1].set_ylabel(r"$\|J_B-J_A\|_1/(2d)$")
for ax in axes:
    ax.set_xlabel(r"Initial $\bar n$")
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax.grid(alpha=.25)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "model_difference.png", dpi=180)
plt.show()

fig, axes = case_axes()
for ax, case in zip(axes.flat, CASE_NAMES):
    for model, (style, color) in MODEL_STYLE.items():
        data = summary.loc[(summary["case"] == case) & (summary["model"] == model)].sort_values("nbar")
        ax.plot(data["nbar"], data["twirl_choi_trace_distance"], style, color=color, label=model)
    ax.set(title=case.replace("_", " "), xlabel=r"Initial $\bar n$", ylabel="Choi distance to Pauli twirl")
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax.grid(alpha=.25)
    ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "pauli_twirl_comparison.png", dpi=180)
plt.show()


## 誤差チャネルの構造

表示する散逸条件は設定セルの `DISPLAY_CASE`、温度は今回計算した最大の $\bar n$ です。
大きな理想成分に埋もれないよう、A・Bは $\chi_{\mathcal N}-\chi_{\mathcal I}$ を表示します。
3列目は $\chi_B-\chi_A$。**差分行列それ自体は量子チャネルではありません。**
各行で色スケールをそろえ、実部と虚部を表示します。

In [ ]:
selected_case = DISPLAY_CASE if DISPLAY_CASE in CASE_NAMES else CASE_NAMES[-1]
selected_nbar = float(max(NBAR_VALUES))
chi_a = channels[("A", selected_case, selected_nbar)]["chi"]
chi_b = channels[("B", selected_case, selected_nbar)]["chi"]
matrices = [chi_a - CHI_IDENTITY, chi_b - CHI_IDENTITY, chi_b - chi_a]
titles = [r"A: $\chi_{\cal N}-\chi_{\cal I}$", r"B: $\chi_{\cal N}-\chi_{\cal I}$", r"$\chi_B-\chi_A$"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
for row, (part, part_label) in enumerate([(np.real, "Real"), (np.imag, "Imaginary")]):
    bound = max(max(float(np.max(np.abs(part(matrix)))) for matrix in matrices), 1e-12)
    for col, (matrix, title) in enumerate(zip(matrices, titles)):
        ax = axes[row, col]
        image = ax.imshow(part(matrix), cmap="RdBu_r", vmin=-bound, vmax=bound)
        ax.set_xticks(range(16), PAULI_LABELS, rotation=90, fontsize=7)
        ax.set_yticks(range(16), PAULI_LABELS, fontsize=7)
        ax.set_title(f"{part_label}: {title}")
    fig.colorbar(image, ax=axes[row, :].tolist(), shrink=.8)
fig.suptitle(f"{selected_case}; nbar={selected_nbar:g}; post-gate error channel")
fig.savefig(OUTPUT_DIR / "chi_structure.png", dpi=180)
plt.show()

completion = {"completed": True, "channels": len(summary), "run_id": RUN_ID}
(OUTPUT_DIR / "completion.json").write_text(encoded(completion), encoding="utf-8")
print(f"Results: {OUTPUT_DIR.resolve()}")


## 結果の読み方と保存物

1. `no_noise` のAは、理想的には初期熱状態に依存しないMSゲートです。残差は時間離散化・運動基底打ち切り等も確認します。
2. Bとの差は、第一サイドバンド結合のフォノン数依存補正の影響です。AもBも、RWAで除いた非共鳴キャリアや第二サイドバンドの作用を評価していません。
3. 散逸を加えた差には、無散逸時の差と散逸による変化の両方が含まれます。ノイズ源の独立性や誤差の単純加算性は仮定しません。
4. $\eta^2 n$ が大きくなる領域ではBの打ち切り精度にも注意が必要です。熱分布の平均だけでなく寄与するFock状態を考えます。
5. 振幅再較正を行っていないため、Bの誤差には較正で抑えられる回転角のずれも含まれます。

出力は `results/ld_second_order_rwa_vs_corrected_sideband/<設定ハッシュ>/` に保存します。
- `config.json`: 全条件・数値設定・コードハッシュ。
- `summary.csv`: モデル別の忠実度、Pauli近似との差、Bell状態、CP/TP診断。
- `model_comparison.csv`: A/B間の差。
- `channels.npz`: 対応ラベル付きの実ゲートsuperoperator・誤差χ・誤差PTM。
- 4枚のPNG: 忠実度、モデル差、Pauli近似との差、χ構造。
- `completion.json`: 全セルの計算・保存が完了したことを示す記録。

既存のシミュレーション、ノートブック、保存済み結果は変更しません。

## 追加比較：モデルC（LD二次展開、運動RWAなし）

**Cは光学RWAを適用済みですが、運動RWAを適用しません。**
指数を絶対次数 $O(\eta^2)$ まで展開したときの、キャリア・第一・第二サイドバンドの全項を残します。
Bの $O(\eta^3)$ 第一サイドバンド補正は加えません。したがってCはBの全項を含むモデルではありません。

| 比較 | 何を変えるか |
|:--|:--|
| A と C | 同じLD二次展開に対して、運動RWAで非共鳴項を捨てるか残すか |
| A と B | 運動RWA後の第一サイドバンドに三次由来の補正を加えるか |
| B と C | LDの保持次数と運動RWAの両方が異なるため、一つの補正の効果とは解釈しない |

2イオンへの結合を等しいとし、相互作用表示で
$$Q(t)=ae^{-i\nu t}+a^\dagger e^{i\nu t},\quad
  S_\rho=\cos\rho\,S_x+\sin\rho\,S_y,\quad
  S_\perp=-\sin\rho\,S_x+\cos\rho\,S_y$$
とすると、実装するCは
$$\boxed{H_C(t)/\hbar=\Omega_0\cos(\mu t)
  \left[S_\perp\left(1-\frac{\eta^2}{2}Q(t)^2\right)
  +\eta S_\rho Q(t)\right].}$$
共通のレーザー位相は $-\pi/2-\rho$、赤青成分の相対位相は0としています。
$S_x=XI+IX$、$S_y=YI+IY$ です。

**A/Bと同じ規格化・離調の符号にするため、$\Omega_0=2g/\eta$、$\mu=\nu-\delta$ とします。**
これに運動RWAを適用すると
$$H_C\longrightarrow\hbar gS_\rho(ae^{-i\delta t}+a^\dagger e^{i\delta t})=H_A$$
となります。以前の説明で使った $\nu+\delta$ とは離調の符号規約が異なります。

Cには、A/BのRWAモデルでは指定不要だった運動周波数が必要です。
初期値は既存の `three_pulse_chi_comparison.py` と同じ **$\nu/(2\pi)=1\,\mathrm{MHz}$** です。
この値は追加のモデル仮定であり、A/Bの既存データから決まるものではありません。
周波数を変更すると別キャッシュになり、A/Bの物理パラメータは変わりません。

前半のimport・設定・plan・functionsセルを先に実行してください。
A/Bの計算セルが未実行でも、以下から同じ設定のA/Bキャッシュを読み込めます。
**Cは高速な項を直接積分するため、A/Bより計算時間が長くなります。**

In [ ]:
# まず no_noise のみで試す場合: C_CASE_NAMES = ["no_noise"]
C_CASE_NAMES = list(CASE_NAMES)
C_NBAR_VALUES = list(NBAR_VALUES)
C_WORKERS = WORKERS
C_REUSE_CACHE = True
C_NUMERICS = {
    "mode_frequency_hz": 1.0e6,
    "fastest_period_steps": 24,
    "solver_atol": 1e-12,
    "solver_rtol": 1e-9,
    "solver_method": "vern9",
}
C_DISPLAY_CASE = DISPLAY_CASE

import ld_second_order_no_motional_rwa as mc
mc = importlib.reload(mc)

assert C_CASE_NAMES and len(set(C_CASE_NAMES)) == len(C_CASE_NAMES)
assert set(C_CASE_NAMES) <= set(case_parameters)
assert C_NBAR_VALUES and len(set(C_NBAR_VALUES)) == len(C_NBAR_VALUES)
assert set(map(float, C_NBAR_VALUES)) <= set(DIMENSIONS)
assert int(C_WORKERS) == C_WORKERS and C_WORKERS >= 1

C_RESOLVED = mc.resolve_parameters(BASE_PARAMETERS, **C_NUMERICS)
display(pd.DataFrame([{
    "mode_frequency_Hz": C_RESOLVED["mode_frequency_hz"],
    "nu_sim": C_RESOLVED["nu_sim"], "mu_sim": C_RESOLVED["mu_sim"],
    "single_tone_Omega0_over_2pi_Hz": C_RESOLVED["omega0_over_2pi_hz"],
    "sideband_detuning_Hz": C_RESOLVED["delta_over_2pi_hz"],
    "drive_offset_from_qubit_Hz": C_RESOLVED["bichromatic_offset_hz"],
    "solver_max_step_sim": C_RESOLVED["solver_options"]["max_step"],
    "solver_max_step_seconds": C_RESOLVED["solver_options"]["max_step"] * C_RESOLVED["seconds_per_sim_unit"],
}]))
print(f"ABC comparison: {3 * len(C_CASE_NAMES) * len(C_NBAR_VALUES)} channels; "
      f"C adds {len(C_CASE_NAMES) * len(C_NBAR_VALUES)} channels")


### Cの積分・保存

高速な係数は解析関数で評価します。従来の500点の線形補間ではサンプリングしません。
最大角周波数 $2\nu+\mu$ の1周期を `fastest_period_steps` 分割するようにODEの最大ステップを制限します。
`fastest_period_steps=48` などに変更して時間方向の収束を比較できます。
運動基底次元・初期熱状態・散逸演算子と物理散逸率はA/Bと共通です。
基底次元の妥当性はCでも別途確認が必要です。

無散逸では全系のユニタリを1回計算し、熱状態を入れて運動を部分トレースします。
散逸ありでは、既存と同じ16入力のLindblad時間発展からチャネルを再構成します。
CはODEの出力状態を再規格化せず、CPTP射影も行いません。
矩形パルス、一定離調・スピン位相、レーザーのランダム揺らぎなしを対象とします。
追加セルは前半の `summary` / `channels` / A/Bの出力フォルダを上書きしません。

In [ ]:
C_ENGINE_HASH = hashlib.sha256(Path(mc.__file__).read_bytes()).hexdigest()
ABC_CONFIG = {
    "schema": "ld-abc-without-motional-rwa-v1", "ab_run_config": RUN_CONFIG,
    "cases": C_CASE_NAMES, "nbar_values": C_NBAR_VALUES,
    "model_c_numerics": C_NUMERICS, "model_c_workers": C_WORKERS,
    "model_c_resolved_baseline": C_RESOLVED,
    "model_c_source_sha256": C_ENGINE_HASH,
    "model_c_definition": "optical RWA; Taylor through absolute eta^2; NO motional RWA",
    "models": {**MODELS, "C": {"label": "C: LD through eta^2, no motional RWA"}},
}
ABC_RUN_ID = hashlib.sha256(encoded(ABC_CONFIG).encode()).hexdigest()[:16]
ABC_OUTPUT_DIR = OUTPUT_ROOT / "with_model_c" / ABC_RUN_ID
C_CACHE_DIR = OUTPUT_ROOT / "model_c_point_cache"
ABC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
C_CACHE_DIR.mkdir(parents=True, exist_ok=True)
(ABC_OUTPUT_DIR / "config.json").write_text(encoded(ABC_CONFIG), encoding="utf-8")

def point_channel_c(case, nbar):
    resolved = mc.resolve_parameters(case_parameters[case], **C_NUMERICS)
    payload = {
        "schema": "ld-c-channel-v1", "parameters": resolved,
        "nbar": float(nbar), "phonon_dim": DIMENSIONS[float(nbar)],
        "workers": C_WORKERS, "qutip_version": qp.__version__,
        "numpy_version": np.__version__, "ms_engine_sha256": ENGINE_HASH,
        "c_engine_sha256": C_ENGINE_HASH,
    }
    signature = encoded(payload)
    key = hashlib.sha256(signature.encode()).hexdigest()
    path = C_CACHE_DIR / f"C_{case}_{key[:24]}.npz"
    if C_REUSE_CACHE and path.exists():
        with np.load(path, allow_pickle=False) as saved:
            if str(saved["signature"].item()) != signature:
                raise RuntimeError(f"Model-C cache signature mismatch: {path}")
            matrix = saved["superoperator"].copy()
            method = str(saved["method"].item())
        if matrix.shape != (16, 16) or not np.isfinite(matrix).all():
            raise RuntimeError(f"Invalid model-C cache: {path}")
        return qp.Qobj(matrix, dims=SUPER_DIMS, superrep="super"), True, method
    result = mc.simulate_channel(
        case_parameters[case], nbar=float(nbar), phonon_dim=DIMENSIONS[float(nbar)],
        parallel_workers=C_WORKERS, **C_NUMERICS,
    )
    channel = result["channel"]
    if not np.isfinite(channel.full()).all():
        raise RuntimeError("Non-finite model-C channel; not cached.")
    temporary = path.with_suffix(".tmp")
    with temporary.open("wb") as stream:
        np.savez_compressed(stream, signature=signature,
                            superoperator=channel.full(), method=result["method"])
    temporary.replace(path)
    return channel, False, result["method"]


In [ ]:
abc_rows = []
abc_channels = {}
abc_started = time.perf_counter()
for case in C_CASE_NAMES:
    for nbar in C_NBAR_VALUES:
        for model in ("A", "B", "C"):
            started_point = time.perf_counter()
            print(f"ABC / {model} / {case}: nbar={nbar:g}, Nv={DIMENSIONS[float(nbar)]}", flush=True)
            if model == "C":
                channel, cached, method = point_channel_c(case, nbar)
            else:
                channel, cached = point_channel(model, case, nbar)
                method = "existing_16_input_qpt"
            values, data = analyze_channel(channel)
            abc_channels[(model, case, float(nbar))] = data
            abc_rows.append({
                "model": model, "case": case, "nbar": float(nbar),
                "phonon_dim": DIMENSIONS[float(nbar)], "from_cache": cached,
                "method": method, "seconds_this_run": time.perf_counter() - started_point,
                **values,
            })
            pd.DataFrame(abc_rows).to_csv(ABC_OUTPUT_DIR / "partial_summary.csv", index=False)
abc_summary = pd.DataFrame(abc_rows).sort_values(["case", "nbar", "model"]).reset_index(drop=True)
if not np.isfinite(abc_summary.select_dtypes(include=np.number).to_numpy()).all():
    raise RuntimeError("Non-finite ABC diagnostics; inspect partial results.")
if (abc_summary["min_choi_eigenvalue"] < -1e-8).any() or (abc_summary["tp_frobenius_error"] > 1e-8).any():
    warnings.warn("ABCのCP/TP診断に残差があります。モデル列ごとの数値設定・収束を確認してください。")
abc_summary.to_csv(ABC_OUTPUT_DIR / "summary.csv", index=False)
display(abc_summary[["model", "case", "nbar", "infidelity", "chi_offdiag_fro", "min_choi_eigenvalue", "tp_frobenius_error"]])
print(f"ABC completed: {len(abc_summary)} channels in {time.perf_counter() - abc_started:.1f} s")


In [ ]:
ABC_PAIRS = [("A", "B"), ("A", "C"), ("B", "C")]
abc_indexed = abc_summary.set_index(["model", "case", "nbar"])
pair_rows = []
for case in C_CASE_NAMES:
    for nbar in C_NBAR_VALUES:
        for left, right in ABC_PAIRS:
            first = abc_channels[(left, case, float(nbar))]
            second = abc_channels[(right, case, float(nbar))]
            pair_rows.append({
                "case": case, "nbar": float(nbar), "pair": left + right,
                "left_model": left, "right_model": right,
                "infidelity_right_minus_left": abc_indexed.loc[(right, case, float(nbar)), "infidelity"] - abc_indexed.loc[(left, case, float(nbar)), "infidelity"],
                "ptm_fro_distance": float(np.linalg.norm(second["ptm"] - first["ptm"])),
                "choi_trace_distance": pcd.trace_norm(second["choi"] - first["choi"]) / 8,
            })
abc_comparison = pd.DataFrame(pair_rows)
abc_comparison.to_csv(ABC_OUTPUT_DIR / "pairwise_comparison.csv", index=False)
np.savez_compressed(
    ABC_OUTPUT_DIR / "channels.npz",
    model=abc_summary["model"].to_numpy(dtype=str), case=abc_summary["case"].to_numpy(dtype=str),
    nbar=abc_summary["nbar"].to_numpy(), pauli_labels=np.asarray(PAULI_LABELS),
    superoperators=np.stack([abc_channels[(r.model, r.case, r.nbar)]["channel"].full() for r in abc_summary.itertuples()]),
    chi_error=np.stack([abc_channels[(r.model, r.case, r.nbar)]["chi"] for r in abc_summary.itertuples()]),
    ptm_error=np.stack([abc_channels[(r.model, r.case, r.nbar)]["ptm"] for r in abc_summary.itertuples()]),
)
display(abc_comparison)

ABC_STYLE = {"A": ("o-", "#0072B2"), "B": ("s--", "#D55E00"), "C": ("^:", "#009E73")}
def abc_case_axes():
    fig, axes = plt.subplots((len(C_CASE_NAMES) + 1) // 2, 2,
                             figsize=(12, 3.5 * ((len(C_CASE_NAMES) + 1) // 2)), squeeze=False)
    for ax in axes.flat[len(C_CASE_NAMES):]:
        ax.set_visible(False)
    return fig, axes

for column, ylabel, filename in [
    ("infidelity", r"$1-F_{\rm avg}$", "abc_infidelity.png"),
    ("twirl_choi_trace_distance", "Choi distance to Pauli twirl", "abc_pauli_twirl.png"),
]:
    fig, axes = abc_case_axes()
    for ax, case in zip(axes.flat, C_CASE_NAMES):
        for model, (style, color) in ABC_STYLE.items():
            data = abc_summary.loc[(abc_summary["case"] == case) & (abc_summary["model"] == model)].sort_values("nbar")
            ax.plot(data["nbar"], data[column], style, color=color, label=model, linewidth=1.8)
        ax.set(title=case.replace("_", " "), xlabel=r"Initial $\bar n$", ylabel=ylabel)
        ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
        ax.grid(alpha=.25)
        ax.legend()
    fig.suptitle(f"A / B / C; mode frequency for C = {C_NUMERICS['mode_frequency_hz']/1e6:g} MHz")
    fig.tight_layout(rect=(0, 0, 1, .96))
    fig.savefig(ABC_OUTPUT_DIR / filename, dpi=180)
    plt.show()

fig, axes = abc_case_axes()
for ax, case in zip(axes.flat, C_CASE_NAMES):
    for left, right in ABC_PAIRS:
        pair = left + right
        data = abc_comparison.loc[(abc_comparison["case"] == case) & (abc_comparison["pair"] == pair)].sort_values("nbar")
        ax.plot(data["nbar"], data["choi_trace_distance"], "o-", label=pair)
    ax.set(title=case.replace("_", " "), xlabel=r"Initial $\bar n$", ylabel="Choi trace distance between models")
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax.grid(alpha=.25)
    ax.legend()
fig.tight_layout()
fig.savefig(ABC_OUTPUT_DIR / "abc_pairwise_distances.png", dpi=180)
plt.show()


In [ ]:
abc_case = C_DISPLAY_CASE if C_DISPLAY_CASE in C_CASE_NAMES else C_CASE_NAMES[-1]
abc_nbar = float(max(C_NBAR_VALUES))
abc_matrices = [abc_channels[(m, abc_case, abc_nbar)]["chi"] - CHI_IDENTITY for m in ("A", "B", "C")]
fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
for row, (part, name) in enumerate([(np.real, "Real"), (np.imag, "Imaginary")]):
    bound = max(max(float(np.max(np.abs(part(matrix)))) for matrix in abc_matrices), 1e-12)
    for col, (model, matrix) in enumerate(zip(("A", "B", "C"), abc_matrices)):
        ax = axes[row, col]
        image = ax.imshow(part(matrix), cmap="RdBu_r", vmin=-bound, vmax=bound)
        ax.set_xticks(range(16), PAULI_LABELS, rotation=90, fontsize=7)
        ax.set_yticks(range(16), PAULI_LABELS, fontsize=7)
        ax.set_title(name + ": " + model + r", $\chi_{\cal N}-\chi_{\cal I}$")
    fig.colorbar(image, ax=axes[row, :].tolist(), shrink=.8)
fig.suptitle(f"ABC error-channel structure: {abc_case}, nbar={abc_nbar:g}")
fig.savefig(ABC_OUTPUT_DIR / "abc_chi_structure.png", dpi=180)
plt.show()
(ABC_OUTPUT_DIR / "completion.json").write_text(encoded({
    "completed": True, "models": ["A", "B", "C"], "channels": len(abc_summary),
    "run_id": ABC_RUN_ID, "amplitude_calibrated": False,
}), encoding="utf-8")
print(f"ABC results: {ABC_OUTPUT_DIR.resolve()}")


### Cの比較で確認すること

- AとCの差は、このLD二次モデルで運動RWAによって落とした非共鳴項の影響です。
- Cも、全次数の光–イオン結合ではありません。CよりBのInfidelityが大きい／小さいことだけでは、B/Cの近似精度の優劣は決まりません。
- Cの周波数依存性は `mode_frequency_hz` で比較できます。$g,\delta,\eta$ を固定して $\nu$ を大きくすると、通常のRWAが良くなる極限でCはAに近づきます。
- A/Bの旧結果とCのODE精度は同一ではありません。微小差を議論する場合は、基底次元・時間刻み・ODE許容誤差の収束を比較します。
- Cでも振幅の再較正は行っていません。Infidelityには較正可能な回転角のずれも含まれます。

出力先は `results/ld_second_order_rwa_vs_corrected_sideband/with_model_c/<設定ハッシュ>/`。
`summary.csv`、`pairwise_comparison.csv`、`channels.npz`、比較PNG4枚、`config.json` を保存します。
Cの点ごとのキャッシュは `model_c_point_cache/` に分離します。